In [0]:
%python
from pyspark.sql.functions import current_timestamp, lit

tables = {
    "patients"   : "patients_csv",
    "doctors"    : "doctors_csv",
    "visits"     : "visits_csv",
    "treatments" : "treatments_csv",
    "claims"     : "claims_csv"
}

def get_all_files(volume_name, table_name):
    files = dbutils.fs.ls(f"/Volumes/healthcare/raw/{volume_name}/")
    matching = [f for f in files if f.name.startswith(table_name) and f.name.endswith(".csv")]
    return matching

spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare.bronze")

for table_name, volume_name in tables.items():
    table_full_name = f"healthcare.bronze.{table_name}"
    files = get_all_files(volume_name, table_name)

    # Get already ingested files
    ingested_files = set()
    if spark.catalog.tableExists(table_full_name):
        ingested_files = set(
            row["dbx_source_file"]
            for row in spark.table(table_full_name)
            .select("dbx_source_file")
            .distinct()
            .collect()
        )

    for file in files:
        file_path = file.path
        file_name = file.name

        # Skip already ingested
        if file_name in ingested_files:
            print(f"Skipping {file_name} (already ingested)")
            continue

        # Read CSV
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "false") \
            .load(file_path)

        # Add metadata
        df = df.withColumn("dbx_ingest_ts", current_timestamp()) \
               .withColumn("dbx_source_file", lit(file_name))

        # Write
        if not spark.catalog.tableExists(table_full_name):
            df.write.format("delta").saveAsTable(table_full_name)
            print(f"Created table: {table_full_name}")
        else:
            df.write.option("mergeSchema", "true").mode("append").saveAsTable(table_full_name)
            print(f"Appended file: {file_name}")

Skipping patients_export_20260425_235320.csv (already ingested)
Skipping patients_export_20260425_235515.csv (already ingested)
Skipping doctors_export_20260425_235321.csv (already ingested)
Skipping doctors_export_20260425_235515.csv (already ingested)
Skipping visits_export_20260425_235321.csv (already ingested)
Skipping visits_export_20260425_235515.csv (already ingested)
Skipping treatments_export_20260425_235321.csv (already ingested)
Skipping treatments_export_20260425_235515.csv (already ingested)
Skipping claims_export_20260425_235321.csv (already ingested)
Skipping claims_export_20260425_235515.csv (already ingested)


In [0]:
-- drop table healthcare.bronze.patients;
-- drop table healthcare.bronze.claims;
-- drop table healthcare.bronze.doctors;
-- drop table healthcare.bronze.treatments;
-- drop table healthcare.bronze.visits;